# Chest X-Ray Pneumonia — Colab Training

Trains the two Part-2 models (`custom_cnn`, `densenet121`) on Google Colab.

**Order of cells**:
1. Confirm GPU
2. Clone repo, install deps
3. Mount Google Drive
4. Download dataset via `kagglehub`
5. Train `custom_cnn` (~15 epochs)
6. Train `densenet121` (~10 epochs)
7. Copy checkpoints + CSV logs to Drive


## 1. Confirm GPU

In [ ]:
!nvidia-smi

## 2. Clone repo and install requirements

In [ ]:
%cd /content
![ -d CS-171-Chest-X-Ray-Medical-Diagnosis ] || git clone https://github.com/jenilkathrotia/CS-171-Chest-X-Ray-Medical-Diagnosis.git
%cd /content/CS-171-Chest-X-Ray-Medical-Diagnosis
!git fetch --all
!git checkout part-2 || git checkout -b part-2 origin/part-2
!git pull origin part-2 || true
!pip install -q -r requirements.txt

## 3. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_OUT = '/content/drive/MyDrive/cs171_chest_xray'
os.makedirs(f'{DRIVE_OUT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_OUT}/logs', exist_ok=True)
print('Drive output dir:', DRIVE_OUT)

## 4. Download dataset via kagglehub

`kagglehub` will cache the dataset; the resulting path contains a `chest_xray/` folder with `train/`, `val/`, `test/`.

In [ ]:
import kagglehub, os

raw_path = kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia')
print('kagglehub root:', raw_path)

# The dataset usually unpacks under .../chest-xray-pneumonia/chest_xray.
candidates = [
    os.path.join(raw_path, 'chest_xray'),
    raw_path,
]
DATA_DIR = next(p for p in candidates if os.path.isdir(os.path.join(p, 'train')))
print('DATA_DIR:', DATA_DIR)
print('classes:', sorted(os.listdir(os.path.join(DATA_DIR, 'train'))))

## 5. Train Custom CNN (~15 epochs)

In [ ]:
import sys
if '/content/CS-171-Chest-X-Ray-Medical-Diagnosis' not in sys.path:
    sys.path.insert(0, '/content/CS-171-Chest-X-Ray-Medical-Diagnosis')

from src.train import train

custom_results = train(
    model_name='custom_cnn',
    data_dir=DATA_DIR,
    epochs=15,
    lr=1e-4,
    batch_size=32,
)
print(custom_results['final_epoch'])
print('checkpoint:', custom_results['checkpoint_path'])

## 6. Train DenseNet121 (~10 epochs)

In [ ]:
densenet_results = train(
    model_name='densenet121',
    data_dir=DATA_DIR,
    epochs=10,
    lr=1e-4,
    batch_size=32,
)
print(densenet_results['final_epoch'])
print('checkpoint:', densenet_results['checkpoint_path'])

## 7. Copy checkpoints and CSV logs to Drive

In [ ]:
import shutil

for name in ['custom_cnn', 'densenet121']:
    ckpt_src = f'results/checkpoints/{name}_best.pt'
    log_src = f'results/logs/{name}.csv'
    if os.path.exists(ckpt_src):
        shutil.copy2(ckpt_src, f'{DRIVE_OUT}/checkpoints/{name}_best.pt')
    if os.path.exists(log_src):
        shutil.copy2(log_src, f'{DRIVE_OUT}/logs/{name}.csv')

print('Drive contents:')
!ls -la {DRIVE_OUT}/checkpoints {DRIVE_OUT}/logs